# General tests
* Specifically - we're testing whether the rows of the consecutive batches overlap (overlapping is determined by *event_id* column)

In [0]:
dbutils.library.restartPython()

In [0]:
from config import tables, create_catalog_and_schema, catalog_name, schema_name

create_catalog_and_schema(catalog_name, schema_name)

bronze_I_tbl = tables['I']['bronze']
silver_I_tbl = tables['I']['silver']

bronze_II_tbl = tables['II']['bronze']
silver_II_tbl = tables['II']['silver']

In [0]:
from data_generators import generate_bronze_data
# Quick test: Generate 2 small batches with different offsets
print("Batch 1 (offset=0) - first 5 event_ids:")
batch1 = generate_bronze_data(NUM_ROWS=10, batch_offset=0)
display(batch1.select("event_id").show())

print("\nBatch 2 (offset=5) - first 5 event_ids:")
batch2 = generate_bronze_data(NUM_ROWS=10, batch_offset=5)
display(batch2.select("event_id").show())

# There are some event_id which occurs in both tables - we're good!!! (and the second table has new records)

# Slowly Changing Dimensions - I type

## Initializing the bronze and silver tables (*the very first batches*)

In [0]:
# Use only if needed and with cautious :D
for tbl in [bronze_I_tbl, silver_I_tbl]:
    if spark.catalog.tableExists(tbl):
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")

In [0]:
from preprocessing import preprocess_new_batch

preprocess_new_batch()

In [0]:
# Reload modules to pick up file changes
import importlib
import data_mergers
import preprocessing
importlib.reload(data_mergers)
importlib.reload(preprocessing)
print("✓ Modules reloaded")

In [0]:
row_count_bronze_I = spark.sql(f"SELECT COUNT(*) FROM {bronze_I_tbl}").collect()[0][0]
print(f"Number of rows in bronze table of SCD type I: {row_count_bronze_I:,}") # Size is as expeceted - 1 million
display(spark.sql(f"SELECT * FROM {bronze_I_tbl} LIMIT 5"))

In [0]:
row_count_silver_I = spark.sql(f"SELECT COUNT(*) FROM {silver_I_tbl}").collect()[0][0]
print(f"Number of rows in silver table of SCD type I: {row_count_silver_I:,}") # 50,000 records less than bronze table
display(spark.sql(f"SELECT * FROM {silver_I_tbl} LIMIT 5"))

## Populating the silver table with new batch of data (*UPSERTING*)

In [0]:
preprocess_new_batch(batch_offset = 750_000)

In [0]:
row_count_bronze_I_after = spark.sql(f"SELECT COUNT(*) FROM {bronze_I_tbl}").collect()[0][0]
print(f"Number of rows in bronze table of SCD type I after ingesting new batch: {row_count_bronze_I_after:,}") # the size is as expected - 2 mln
display(spark.sql(f"SELECT * FROM {bronze_I_tbl} LIMIT 5"))

In [0]:
row_count_silver_I_after = spark.sql(f"SELECT COUNT(*) FROM {silver_I_tbl}").collect()[0][0]
print(f"Number of rows in silver table of SCD type I: {row_count_silver_I_after:,}") # 1.7 mln records - some records from the previous version of the silver table were updated with the new batch and therefore the size isn't 2 mln
display(spark.sql(f"SELECT * FROM {silver_I_tbl} LIMIT 5"))

# Slowly Changing Dimensions - II type

In [0]:
# Use only if needed and with cautious :D
for tbl in [bronze_II_tbl, silver_II_tbl]:
    if spark.catalog.tableExists(tbl):
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")

In [0]:
preprocess_new_batch(scd_type="II")

In [0]:
row_count_bronze_II = spark.sql(f"SELECT COUNT(*) FROM {bronze_II_tbl}").collect()[0][0]
print(f"Number of rows in bronze table of SCD type II: {row_count_bronze_II:,}") # Size is as expeceted - 1 million
display(spark.sql(f"SELECT * FROM {bronze_II_tbl} LIMIT 5"))

* We deliberately skip showing the basic stats for silver table SCD II because nothing new and interesting has happened there - it's the same as in the SCD I approach.

In [0]:
preprocess_new_batch(batch_offset = 750_000, scd_type="II")

* Again, for the sake of your prrecious time, we deliberately skip showing the basic stats for bronze table SCD II after the second batch ingestion because nothing new and interesting has happened there - it's the same as in the SCD I approach.

In [0]:
row_count_silver_II_after = spark.sql(f"SELECT COUNT(*) FROM {silver_II_tbl}").collect()[0][0]
print(f"Number of rows in silver table of SCD type II: {row_count_silver_II_after:,}") 
display(spark.sql(f"SELECT * FROM {silver_II_tbl} LIMIT 5"))

In [0]:
display(spark.sql(f"SELECT `is_current` AS `current_state`, COUNT(*) FROM {silver_II_tbl} GROUP BY `is_current`")) # there are both active and inactive records

* One final test for SCD II - we're going to ingest another, third batch and see what happens to the records of the same id. Specifically, will the new record update the oldest, inactive records? Just making sure the data are correct and not distorted - the merging condition was updated to update only active records, but this caveat is something one should be aware of - do not "update" oldest records

In [0]:
preprocess_new_batch(batch_offset = 850_000, scd_type="II")

In [0]:
from pyspark.sql.functions import col

filtering_query = f"""
SELECT event_id
FROM {silver_II_tbl}
GROUP BY event_id
HAVING COUNT(*) >= 3
"""

df = spark.sql(f"""
SELECT event_id, event_timestamp, valid_from, valid_to, is_current, device_type, genre
FROM {silver_II_tbl}
WHERE event_id IN (
    {filtering_query}
)
""")
df_sort = df.sort(col("event_id"))
display(df_sort)

* As can be seen - the SCD works as expected.

# Schema Enforcemenet

* The test: we're generating another batch of bronze data (as previously) and we take two approaches to test the *schema enforcemenet* mechanism:
- First: we manually add a new column
- Second: we stay with same columns but cast a numeric column to a string, then add some non-numeric values to each record.

Either case in the end we attempt to write the new batch to the existing bronze table

### Extra column mismatch

In [0]:
SE_batch1 = generate_bronze_data(NUM_ROWS = 5_000)
display(SE_batch1.show(5))

In [0]:
from pyspark.sql.functions import lit

SE_batch1_ext = SE_batch1.withColumn("country_extended", lit("Poland"))
display(SE_batch1_ext.show(5))

In [0]:
%skip
SE_batch1_ext.write.format("delta").mode("append").saveAsTable(bronze_I_tbl)
# [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table. SQLSTATE: 42KDG
# This is the error we wanted and expected ;D

![DELTA_METADATA_MISMATCH - Extra Column Error](./images/error_delta_metadata_mismatch_extra_column.png "Error when attempting to write DataFrame with extra column to existing Delta table")

### Mismatched datatype

In [0]:
SE_batch2 = generate_bronze_data(NUM_ROWS = 5_000)

In [0]:
from pyspark.sql.functions import concat

SE_batch2_ext = SE_batch2.withColumn("play_time_seconds", concat(col("play_time_seconds"), lit("_xD")))
display(SE_batch2_ext.show(5))

In [0]:
SE_batch2_ext.printSchema()

In [0]:
%skip
SE_batch2_ext.write.format("delta").mode("append").saveAsTable(bronze_I_tbl)
# [CAST_INVALID_INPUT]  -another error we wanted and expected :DDD

![CAST_INVALID_INPUT - String to Int Error](./images/error_cast_invalid_input_string_to_int.png "Error when attempting to cast string column with non-numeric values to integer type")

# Schema Evolution

* Now let's relax Schema Enforcement mechanism and attempd to ingest the new batch with extra column into the bronze table

## Extra columns

In [0]:
SEV_batch1 = generate_bronze_data(NUM_ROWS = 5_000)
SEV_batch1_ext = SEV_batch1.withColumn("country_extended",  lit("USA"))


In [0]:
SEV_batch1_ext.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(bronze_I_tbl) # Worked nice and clean

In [0]:
bronze_I_table = spark.read.table(bronze_I_tbl)

null_count = bronze_I_table.filter(col("country_extended").isNull()).count()
total_count = bronze_I_table.count()

print(f"Null count: {null_count} ({round(null_count / total_count * 100, 2)}%)") # Nearly all rows have missing values in the extra column since this field hadn't existed before the insertion.

## Data mismatch

In [0]:
SEV_batch2 = generate_bronze_data(NUM_ROWS = 5_000)

In [0]:
SEV_batch2_ext = SEV_batch2.withColumn("play_time_seconds", concat(col("play_time_seconds"), lit("_xD")))
SEV_batch2_ext2 = SEV_batch2.withColumn("play_time_seconds", col("play_time_seconds").cast("string"))


In [0]:
%skip
SEV_batch2_ext.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(bronze_I_tbl) # Didn't work. 
# Databricks cannot cast the column to string which is the common type of Integer and String

![Schema Merge Incompatible Types Error](./images/error_schema_merge_incompatible_types.png "Error when mergeSchema=true cannot find common type between integer and string with non-numeric values")

In [0]:
SEV_batch2_ext2.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(bronze_I_tbl) # This time writing string column worked because `play_time_seconds` is a numeric string column

## Renamed column error?
* I'm really curious if only renaming a column can trigger Schema Mismatch Error. Let's find out

In [0]:
renamed_col_batch = generate_bronze_data(NUM_ROWS = 5_000)
renamed_col_batch = renamed_col_batch.withColumnRenamed("event_timestamp", "event_time")

In [0]:
%skip
renamed_col_batch.write.format("delta").mode("append").saveAsTable(bronze_I_tbl) # Even a simple column renaming can cause schema mismatch

![Schema Mismatch - Renamed Column Error](./images/error_schema_mismatch_renamed_column.png "Error when attempting to write DataFrame with renamed column to existing Delta table")

In [0]:
display(spark.sql(f"DESCRIBE TABLE EXTENDED {bronze_I_tbl}"))

## Type Widening
To test type widening, we'll do the simple workflow:
1. Generate a batch
2. Cast an INTEGER column to BigInt column and scale up the values so that the BigInt is actually need to store them
3. Write the batch to bronze table

In [0]:
TW_batch = generate_bronze_data(NUM_ROWS = 5_000)

In [0]:
from pyspark.sql.types import LongType

TW_batch_widened = TW_batch.withColumn("play_time_seconds", col("play_time_seconds").cast(LongType()))

In [0]:
TW_batch_widened.printSchema()

In [0]:
multiplier = 2147483647
TW_batch_widened = TW_batch_widened.withColumn("play_time_seconds", multiplier*col("play_time_seconds"))

TW_batch = TW_batch.withColumn("play_time_seconds_overflowed", multiplier*col("play_time_seconds")) # Just for curiosity we want to see how pyspark reacts when it encounters number overflow

In [0]:
%skip
TW_batch.select('play_time_seconds_overflowed').show(5)

![Integer Overflow Error](./images/error_integer_overflow.png "Error showing integer overflow when multiplying by 2147483647")

In [0]:
TW_batch_widened.select("play_time_seconds").show(5) # overflown contained :)

In [0]:
%skip
TW_batch_widened.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(bronze_I_tbl)

Well, it's strange. It seems type widening doesn't work. I'd suggest calling the below reaction as "type narowing"
![Type Widening/Narrowing Error](./images/error_type_widening_narrowing.png "Error when attempting type widening from int to long - Delta requires explicit type widening enablement")

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {bronze_I_tbl}")) # the MinWriterVersion is 7 so type widening should be enabled by default, but not?
# we'll change the metadata manually

* Changing the table properties manually didn't solve the problem.

In [0]:
spark.sql(f"ALTER TABLE {bronze_I_tbl} SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")

In [0]:
TW_batch_widened.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(bronze_I_tbl)

# Column mapping: renaming and dropping

In [0]:
spark.sql(f"""
ALTER TABLE {silver_I_tbl} SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
)
""")

In [0]:
spark.sql(f"""
ALTER TABLE {silver_I_tbl} RENAME COLUMN event_timestamp TO event_time
""") # Renaming works as expected

In [0]:
display(spark.sql(f"DESCRIBE TABLE {silver_I_tbl}"))

In [0]:
spark.sql(f"""
ALTER TABLE {silver_I_tbl} DROP COLUMN royalty_rate
""")

In [0]:
display(spark.sql(f"DESCRIBE TABLE {silver_I_tbl}")) # 'royalty_rate' column is gone

Just for curiosity we'll see what happens if we don't set 'delta.columnMapping.mode' = 'name' but leave the default value (which is None) and try renaming a column

In [0]:
%sql
CREATE TABLE test_table (
  id INT,
  ugly_column STRING
) USING DELTA;

INSERT INTO test_table VALUES (1, 'Test');

In [0]:
%skip
%sql
ALTER TABLE test_table RENAME COLUMN ugly_column TO nice_column; -- We got an expected error, nice :D

![Column Mapping Mode Required Error](./images/error_column_mapping_mode_required.png "Error when attempting to rename column without enabling delta.columnMapping.mode")

In [0]:
%sql
DROP TABLE IF EXISTS test_table;


# Controlled Evolution & Data Contracts: The "Tap & Sewage Filter" Analogy

## Why Silent Schema Evolution Is Risky

- **Silent schema evolution (`mergeSchema = true`) in production:**
  - Upstream teams may introduce breaking changes:
    - Renaming fields (e.g., `user_id` → `user_identification`)
    - Dropping critical fields without notice
  - **Result:** Silent evolution blindly accepts these mutations
    - Corrupted Silver tables
    - Blank dashboards for stakeholders

---

## Solution: Data Contracts & Quarantine Pattern

### 1. Data Contracts

- **Explicit schema definitions**
- **Data quality rules**
- **Ingestion Gate (Contract Check):**
  - Validates incoming records against contracts

### 2. Quarantine (Dead Letter Queue) Pattern

- **Valid records:**
  - Proceed to Silver layer
  - No breaking downstream dependencies
- **Malformed/contract-breaking records:**
  - Diverted into quarantine table (`silver_quarantine`)

### 3. Collaborative Remediation

- **Alert triggers:**
  - Notifies data engineers
  - Collaboration with upstream source owners
  - Investigate root causes (e.g., missing keys, unexpected drops)
  - Update contract via Git PR
  - Reprocess quarantined records

---

## Analogy: Tap & Sewage Filter

- **Washing dishes at a sink:**
  - If untreated sewage flows from the tap:
    - Shut off the valve to protect clean dishes
    - Install a bypass filter:
      - Clean water flows to dishes
      - Sewage diverted to holding tank
      - Filtered/processed water safely rejoins system

---

> (Prosze mi wierzyć, że te analogię z szambem i zlewem wymyśliłem sam - to nie było AI :). AI jedynie sformatowało tekst, żeby przyjemnie się czytało.

# Reliability & table maintenance 

## OPTIMIZE, so compressing multiple, small parquet files into bigger ones

In [0]:
display(spark.sql(f"OPTIMIZE {silver_I_tbl}"))

In [0]:
display(spark.sql(f"OPTIMIZE {silver_II_tbl}"))

## VACUUM - deleting older (smaller) files

In [0]:
display(spark.sql(f"VACUUM {silver_I_tbl} RETAIN 168 HOURS")) # Nothing was, I suppose, expected to happen because the table is relatively fresh (it's been here for a few minutes).

In [0]:
display(spark.sql(f"VACUUM {silver_II_tbl} RETAIN 168 HOURS"))

## Partitioning

### PARTITION BY

In [0]:
%sql
CREATE TABLE test_partitioned (
  id INT,
  event_date DATE,
  amount DOUBLE
) USING DELTA
PARTITIONED BY (event_date); -- The PARTTION BY command is now shadowed which should tell you it's now recommended. Nice detail

In [0]:
%sql
INSERT INTO test_partitioned VALUES (1, '2022-01-01', 100.0);

In [0]:
%sql
DESCRIBE EXTENDED test_partitioned;

In [0]:
%sql
DROP TABLE test_partitioned;

### ZORDER BY

In [0]:
%sql
CREATE TABLE test_zorder (
  user_id STRING,
  device_type STRING,
  click_count INT
) USING DELTA;

In [0]:
%sql
INSERT INTO test_zorder VALUES 
  ('usr_1', 'mobile', 5), 
  ('usr_2', 'desktop', 10), 
  ('usr_1', 'tablet', 2),
  ('usr_3', 'mobile', 8);

In [0]:
%sql
OPTIMIZE test_zorder ZORDER BY (user_id, device_type);

In [0]:
%sql
DROP TABLE test_zorder;

### LIQUID CLUSTERING

In [0]:
%sql
CREATE TABLE liquid_test_table (
  id STRING,
  event_time TIMESTAMP,
  temperature DOUBLE,
  device_id STRING
) USING DELTA
CLUSTER BY (device_id, event_time);

In [0]:
%sql
DESCRIBE EXTENDED liquid_test_table;

In [0]:
%sql
DROP TABLE liquid_test_table;

In [0]:
spark.sql(f"ALTER TABLE {silver_I_tbl} CLUSTER BY (genre)")
spark.sql(f"OPTIMIZE {silver_I_tbl} FULL")

In [0]:
display(spark.sql(f"DESCRIBE EXTENDED {silver_I_tbl}"))

## Data Quality Rules
We'll apply the following data quality constrains:
- event_id musn't be NULL and (with event_time (formerly: event_timestamp)) has to be unique
- play_time_seconds has to be non-negative


In [0]:
spark.sql(f"ALTER TABLE {silver_I_tbl} DROP CONSTRAINT IF EXISTS unique_event_id_event_time")
spark.sql(f"ALTER TABLE {silver_I_tbl} DROP CONSTRAINT IF EXISTS play_time_non_negative")
spark.sql(f"ALTER TABLE {silver_I_tbl} ALTER COLUMN event_time DROP NOT NULL")

In [0]:
spark.sql(f"ALTER TABLE {silver_I_tbl} ALTER COLUMN event_time SET NOT NULL")

spark.sql(f"""
    ALTER TABLE {silver_I_tbl} 
    ADD CONSTRAINT play_time_non_negative 
    CHECK (play_time_seconds >= 0)
""")

spark.sql(f"""
    ALTER TABLE {silver_I_tbl}
    ADD CONSTRAINT unique_event_id_event_time
    UNIQUE (event_id, event_time)
""")

In [0]:
display(spark.sql(f"DESCRIBE EXTENDED {silver_I_tbl}"))

# Optional task

### 1. Configuration as Code (YAML)
A data contract is implemented as a physical configuration file, typically YAML, stored directly within the project repository (e.g., using Databricks Asset Bundles). Instead of relying on passive text documentation, this file defines strict rules, such as expected columns, data types, and non-null constraints. The Databricks pipeline parses this file at runtime, creating an automated validation gate that checks all incoming data against these rules before ingestion.

---

### 2. Managing Upstream Changes via Pull Requests
Schema changes are no longer deployed without prior notice. If an upstream application team needs to add or modify a column, they are required to update the YAML contract and submit a Pull Request (PR). This workflow ensures that Data Engineers review, discuss, and approve the modifications before they reach the main branch, preventing unexpected pipeline failures in the production environment.

---

### 3. Dead Letter Queue (Quarantine) Recovery
Records that fail the contract validation are diverted to a quarantine table (Dead Letter Queue). We never delete this data, as it holds business value. Instead, we apply recovery logic: missing values (such as a missing primary key) can be backfilled using joins with reference tables, and valid structural changes (like a column rename) can be resolved by updating the metadata using Delta Column Mapping. Once the data is corrected and meets the contract requirements, it is reprocessed and loaded into the Silver layer.